# La portée des Logic Tensor Networks

**Préambule** : il est recommandé de parcourir les tutoriels (dossier `/tutorial`) avant d'aborder ces exemples.

L'objectif de cette série d'exemples est de montrer comment le langage de Real Logic permet de spécifier un grand nombre de tâches impliquant à la fois de l'apprentissage à partir de données et du raisonnement logique. La classification, la régression, le clustering, ou la prédiction de liens en sont autant d'exemples : dans tous les cas, ce sont toujours les mêmes briques (constantes, variables, prédicats, connecteurs, quantificateurs) qui sont réutilisées, seule la spécification logique du problème change.

La solution d'un problème spécifié en Real Logic s'obtient en interprétant cette spécification via un Logic Tensor Network. La bibliothèque LTN implémente Real Logic en PyTorch : chaque opérateur logique est groundé à l'aide de primitives PyTorch, de sorte que LTN construit directement un graphe de calcul PyTorch. Grâce à l'optimisation native de PyTorch, LTN reste relativement efficace tout en offrant l'expressivité de la logique du premier ordre.

Le premier exemple que nous allons voir repose sur l'une des tâches d'apprentissage automatique les plus simples et les plus intuitives : la classification binaire.

## Classification binaire

La tâche d'apprentissage automatique la plus simple est la classification binaire. Supposons que l'on souhaite apprendre un classifieur binaire $A$ pour un ensemble de points de $[0,1]^2$, à partir d'exemples positifs et négatifs déjà connus.

Avant de construire ce problème en code, spécifions-le complètement, en réutilisant le vocabulaire déjà établi dans les tutoriels, complété par deux nouvelles notations : $D(\cdot)$, qui indique le domaine d'une variable ou d'une constante, et $D_{in}(\cdot)$, qui indique le domaine attendu en entrée d'une fonction ou d'un prédicat.

**Domaines :**
- $points$ (désignant les exemples).

**Variables :**
- $x_+$ pour les exemples positifs ;
- $x_-$ pour les exemples négatifs ;
- $x$ pour l'ensemble des exemples ;
- $D(x) = D(x_+) = D(x_-) = points$ : les trois variables puisent leurs valeurs dans le même domaine.

**Prédicats :**
- $A(x)$, le classifieur entraînable ;
- $D_{in}(A) = points$ : le prédicat $A$ attend en entrée un objet du domaine $points$.

**Axiomes :**
- $\forall x_+\, A(x_+)$ : le prédicat doit être vrai pour les exemples positifs ;
- $\forall x_-\, \lnot A(x_-)$ : le prédicat doit être faux pour les exemples négatifs.

Contrairement à l'apprentissage supervisé classique, qui minimiserait un écart entre une prédiction et une étiquette, ces deux axiomes traduisent directement l'objectif d'apprentissage sous forme de règles logiques à satisfaire, exactement dans l'esprit de ce qu'on avait déjà vu avec l'exemple de classification par proximité, mais ici la vérité terrain est connue pour tous les exemples, pas seulement pour deux points de référence.

**Grounding :**
- $\mathcal{G}(points)=[0,1]^2$ : ici, le grounding ne pointe pas vers un individu précis, mais vers l'espace entier d'où proviennent les points, c'est le grounding du domaine lui-même ;
- $\mathcal{G}(x) \in [0,1]^{m\times2}$ : $\mathcal{G}(x)$ est une séquence de $m$ points, c'est-à-dire $m$ exemples, exactement le mécanisme de variable vu depuis le premier tutoriel ;
- $\mathcal{G}(x_+) = \langle d\in\mathcal{G}(x) \mid \|d-(0.5,0.5)\|<0.09\rangle$ : $\mathcal{G}(x_+)$ regroupe, par construction, les exemples d'entraînement dont la distance euclidienne au centre $(0.5,0.5)$ est inférieure au seuil $0.09$ ;
- $\mathcal{G}(x_-) = \langle d\in\mathcal{G}(x) \mid \|d-(0.5,0.5)\|\geq0.09\rangle$ : $\mathcal{G}(x_-)$ regroupe, symétriquement, les exemples dont la distance au centre est supérieure ou égale à ce même seuil ;
- $\mathcal{G}(A\mid\theta): x\mapsto \sigma(\text{MLP}_\theta(x))$, où $\text{MLP}$ est un perceptron multicouche à une seule sortie, dont les paramètres $\theta$ doivent être appris, exactement la construction déjà rencontrée pour tout prédicat entraînable.

### Jeu de données

Définissons maintenant notre jeu de données jouet, avec des points dans $[0,1]^2$. On génère aléatoirement 100 points, et on leur attribue une étiquette selon le grounding défini plus haut : les points proches du centre seront classés positifs, ceux éloignés du centre seront classés négatifs.

In [ ]:
import torch
import matplotlib.pyplot as plt

nr_samples = 100
dataset = torch.rand((nr_samples, 2))
labels_dataset = torch.sum(torch.square(dataset - torch.tensor([.5, .5])), dim=1) < .09

Traçons notre jeu de données pour observer que les points proches du centre reçoivent une étiquette positive, tandis que les points éloignés du centre reçoivent une étiquette négative.

In [ ]:
plt.figure(figsize=(4,4))
plt.scatter(dataset[labels_dataset][:,0], dataset[labels_dataset][:,1], label='A')
plt.scatter(dataset[torch.logical_not(labels_dataset)][:,0], dataset[torch.logical_not(labels_dataset)][:,1], label='~A')
plt.title("Ground truth")
plt.legend()
plt.show()

### Paramétrage LTN

Pour définir notre base de connaissances (les axiomes), il faut définir le prédicat $A$, les connecteurs, les quantificateurs, et l'opérateur `SatAgg`.

Pour les connecteurs et quantificateurs, on utilise la configuration produit stable, déjà présentée dans les tutoriels. Pour le prédicat $A$, on utilise un simple perceptron multicouche (MLP), implémenté comme un `torch.nn.Module`, exactement la même construction rencontrée à plusieurs reprises : des couches linéaires, une activation ELU entre elles, et une sigmoïde finale garantissant une sortie dans $[0,1]$.

`SatAgg`, l'agrégateur de satisfaction déjà rencontré dans le tutoriel sur l'apprentissage, agrège les degrés de vérité de toutes les formules closes de la base de connaissances. Par défaut, il est implémenté avec l'agrégateur `pMeanError`.

Ici, seuls `Not` et `Forall` sont nécessaires, puisque les deux axiomes de $\mathcal{K}$ ($\forall x_+\,A(x_+)$ et $\forall x_-\,\lnot A(x_-)$) ne font intervenir que ces deux opérateurs.

In [ ]:
import ltn
# we define predicate A
class ModelA(torch.nn.Module):
    def __init__(self):
        super(ModelA, self).__init__()
        self.sigmoid = torch.nn.Sigmoid()
        self.layer1 = torch.nn.Linear(2, 16)
        self.layer2 = torch.nn.Linear(16, 16)
        self.layer3 = torch.nn.Linear(16, 1)
        self.elu = torch.nn.ELU()

    def forward(self, x):
        x = self.elu(self.layer1(x))
        x = self.elu(self.layer2(x))
        return self.sigmoid(self.layer3(x))


A = ltn.Predicate(ModelA())

# we define the connectives, quantifiers, and the SatAgg
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
SatAgg = ltn.fuzzy_ops.SatAgg()

### Utilitaires

Définissons maintenant quelques classes et fonctions utilitaires.

On définit un chargeur de données PyTorch standard, qui prend en entrée le dataset et renvoie un générateur de batchs. On a besoin de deux instances distinctes : une pour les données d'entraînement, une pour les données de test. 50 exemples sont réservés à l'entraînement, 50 autres au test, afin de vérifier que le modèle généralise bien au-delà des exemples qu'il a vus pendant l'apprentissage.

On définit ensuite deux fonctions pour évaluer les performances du modèle, chacune calculée sur l'ensemble de test :
* le niveau de satisfaction de la base de connaissances, qui mesure la capacité de LTN à satisfaire les règles logiques ;
* la précision de classification, qui mesure directement la qualité des prédictions par rapport aux vraies étiquettes.

Pour `compute_sat_level`, on sépare, pour chaque batch, les exemples positifs des exemples négatifs, on reconstruit les deux variables correspondantes, puis on évalue les deux axiomes de $\mathcal{K}$ exactement comme ils ont été spécifiés plus haut, avant de les agréger via `SatAgg`. Ce résultat est moyenné sur l'ensemble des batchs du chargeur. Un détail technique mérite d'être signalé : la sélection des exemples positifs et négatifs se fait ici via `torch.nonzero`, qui introduit une dimension supplémentaire inhabituelle dans le tenseur obtenu (une forme $(k,1,2)$ plutôt que $(k,2)$), contrairement à l'indexation booléenne directe utilisée plus haut dans ce notebook. Cette différence n'a toutefois aucun effet sur le résultat final : LTN gère cette dimension supplémentaire en interne, et les deux façons d'indexer produisent des valeurs de vérité rigoureusement identiques.

Pour `compute_accuracy`, on accède directement au réseau PyTorch brut sous-jacent au prédicat $A$, via `A.model`, en lui passant les données telles quelles plutôt que sous forme de variable LTN. Cela permet d'obtenir directement les probabilités prédites par le réseau, sans passer par toute la mécanique LTN (`LTNObject`, `free_vars`), puisqu'on ne cherche ici qu'à calculer une métrique de classification standard, pas à évaluer une formule logique. Les probabilités sont ensuite seuillées à $0.5$ pour obtenir des décisions binaires, comparées aux vraies étiquettes via la métrique `accuracy_score` de `scikit-learn`.

Enfin, on construit les deux chargeurs de données : les 50 premiers points pour l'entraînement, les 50 derniers pour le test. La taille de batch choisie, 64, étant supérieure au nombre de points disponibles, chaque chargeur ne produit en réalité qu'un seul batch par epoch. Les données d'entraînement sont mélangées à chaque parcours, contrairement aux données de test, pour lesquelles ce mélange n'a aucune utilité puisqu'elles ne servent qu'à l'évaluation.

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

# this is a standard PyTorch DataLoader to load the dataset for the training and testing of the model
class DataLoader(object):
    def __init__(self,
                 data,
                 labels,
                 batch_size=1,
                 shuffle=True):
        self.data = data
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.data.shape[0] / self.batch_size))

    def __iter__(self):
        n = self.data.shape[0]
        idxlist = list(range(n))
        if self.shuffle:
            np.random.shuffle(idxlist)

        for _, start_idx in enumerate(range(0, n, self.batch_size)):
            end_idx = min(start_idx + self.batch_size, n)
            data = self.data[idxlist[start_idx:end_idx]]
            labels = self.labels[idxlist[start_idx:end_idx]]

            yield data, labels


# define metrics for evaluation of the model

# it computes the overall satisfaction level on the knowledge base using the given data loader (train or test)
def compute_sat_level(loader):
    mean_sat = 0
    for data, labels in loader:
        x_A = ltn.Variable("x_A", data[torch.nonzero(labels)])  # positive examples
        x_not_A = ltn.Variable("x_not_A",
                               data[torch.nonzero(torch.logical_not(labels))])  # negative examples
        mean_sat += SatAgg(
            Forall(x_A, A(x_A)),
            Forall(x_not_A, Not(A(x_not_A)))
        )
    mean_sat /= len(loader)
    return mean_sat

# it computes the overall accuracy of the predictions of the trained model using the given data loader
# (train or test)
def compute_accuracy(loader):
    mean_accuracy = 0.0
    for data, labels in loader:
        predictions = A.model(data).detach().numpy()
        predictions = np.where(predictions > 0.5, 1., 0.).flatten()
        mean_accuracy += accuracy_score(labels, predictions)

    return mean_accuracy / len(loader)

# create train and test loader, 50 points each
# batch size is 64, meaning there is only one batch for epoch
train_loader = DataLoader(dataset[:50], labels_dataset[:50], 64, True)
test_loader = DataLoader(dataset[50:], labels_dataset[50:], 64, False)

### Apprentissage

Notons $D$ l'ensemble complet des exemples du dataset. L'objectif, pour la base de connaissances $\mathcal{K}=\{\forall x_+\,A(x_+),\ \forall x_-\,\lnot A(x_-)\}$, est donné par $\text{SatAgg}_{\phi\in\mathcal{K}}\ \mathcal{G}_{\theta,\,x\leftarrow D}(\phi)$.

Décomposons cette notation, qui rassemble en un seul symbole deux dépendances qu'il faut toujours garder à l'esprit pour évaluer numériquement une formule LTN.

**La dépendance en $\theta$.** Le prédicat $A$ n'est pas une formule mathématique figée, c'est un réseau de neurones dont les poids changent au fil de l'entraînement. Prenons un point $q$ précis : avant l'entraînement, avec des poids initiaux $\theta_0$ tirés au hasard, le réseau pourrait donner $A(q)=0.5$, une confiance médiocre, aucune connaissance encore acquise. Après 1000 epochs, avec des poids ajustés $\theta_{1000}$, ce même point $q$ pourrait donner $A(q)=0.92$. C'est exactement le même symbole $A$, le même point $q$, mais le résultat change du tout au tout selon quels poids on utilise pour évaluer $A$. L'indice $\theta$ précise donc, à chaque instant, avec quel jeu de poids le grounding est calculé.

**La dépendance en $x\leftarrow D$.** Une variable comme $x_+$ ne dit rien, en elle-même, sur quels individus concrets la remplissent : c'est au moment du grounding qu'on décide quel tenseur lui est associé. La notation $x\leftarrow D$ se lit littéralement comme une affectation, "$x$ reçoit les valeurs de $D$", exactement comme une ligne de code `x = D` donnerait une valeur concrète à une variable.

$\mathcal{G}_{\theta,\,x\leftarrow D}(\phi)$ se lit donc : le degré de vérité de la formule $\phi$, calculé avec les poids actuels $\theta$ du réseau, en remplissant la variable $x$ avec les données du dataset $D$.

En pratique, l'optimiseur utilise la fonction de perte suivante :

$$L = 1-\text{SatAgg}_{\phi\in\mathcal{K}}\ \mathcal{G}_{\theta,\,x\leftarrow B}(\phi)$$

où $B$ est un mini-batch tiré de $D$. La seule différence avec l'objectif théorique est que $x$ n'est plus rempli avec le dataset complet, mais avec un sous-ensemble $B$ : recalculer la satisfaction sur l'ensemble complet à chaque étape de gradient serait trop coûteux, donc on l'approxime sur un batch, quitte à changer de batch à chaque itération pour couvrir tout le dataset au fil des epochs, exactement le principe déjà mis en pratique avec le `DataLoader`.

En minimisant cette perte, on cherche donc à maximiser l'opérateur `SatAgg` appliqué à la base de connaissances, c'est-à-dire à maximiser le niveau de satisfaction de chaque formule qu'elle contient.

Cet objectif et cette perte dépendent de plusieurs choix, qu'on a déjà détaillés en profondeur dans les tutoriels :
* le choix de la sémantique floue utilisée pour approximer chaque connecteur et quantificateur ;
* le choix des hyperparamètres internes de ces opérateurs, comme la valeur de l'exposant $p$ dans une moyenne généralisée ;
* le choix de la fonction d'agrégation des formules, `SatAgg`.

Dans ce qui suit, on entraîne notre LTN sur la tâche de classification binaire, en utilisant la satisfaction de la base de connaissances comme objectif. Autrement dit, on cherche à apprendre les paramètres $\theta$ du prédicat unaire $A$ de façon à satisfaire au mieux les deux axiomes de la base de connaissances. Le modèle est entraîné sur 1000 epochs, avec l'optimiseur `Adam`.

La figure suivante montre le graphe de calcul LTN associé à cette tâche.

![Graphe de calcul](./images/binary-classification.png)

---

### Lecture complète du schéma, étape par étape

Ce schéma est une représentation graphique exacte de la ligne de code suivante :

```python
sat_agg = SatAgg(
    Forall(x_A, A(x_A)),
    Forall(x_not_A, Not(A(x_not_A)))
)
```

Reprenons-le de gauche à droite, chaque bloc correspondant à une étape précise de ce calcul.

#### Étape 1 : les entrées, $\mathcal{G}(x_+)$ et $\mathcal{G}(x_-)$

Les deux rectangles à gauche représentent le grounding des deux variables : $\mathcal{G}(x_+)$ est le batch de points positifs, $\mathcal{G}(x_-)$ le batch de points négatifs, exactement les tenseurs construits en code avec `x_A = ltn.Variable(...)` et `x_not_A = ltn.Variable(...)`. Ce sont de simples données d'entrée, rien n'est encore calculé à ce stade.

#### Étape 2 : le réseau partagé, $\mathcal{G}_\theta(A)$

Le bloc violet au centre représente le prédicat $A$, le MLP défini dans `ModelA`, paramétré par ses poids $\theta$. C'est le même bloc, donc le même réseau avec les mêmes poids, qui traite à la fois $x_+$ et $x_-$. Ce n'est pas un réseau différent pour chaque classe, c'est un seul prédicat $A(x)$ qu'on applique deux fois, une fois à chaque groupe de points, ce qui correspond à `A(x_A)` et `A(x_not_A)` dans le code.

#### Étape 3 : les sorties du réseau, $\mathcal{G}_\theta(A(x_+))$ et $\mathcal{G}_\theta(A(x_-))$

Après passage dans le réseau, chaque point de $x_+$ obtient sa propre sortie individuelle du prédicat $A$, une valeur dans $[0,1]$. La sortie complète pour $x_+$ est donc un tenseur d'autant de valeurs que de points dans $x_+$ Il en va de même pour $x_-$. Aucune agrégation n'a encore eu lieu à ce stade.

#### Étape 4 : la négation, $N_S$

Le cercle marqué $N_S$ (négation standard, $\lnot u = 1-u$), présent uniquement sur la branche $x_-$, traduit le fait que l'axiome sur les négatifs contient une négation ($\forall x_-\,\lnot A(x_-)$). Dans le code, cela correspond à  l'appel `Not(A(x_not_A))`.

#### Étape 5 : les deux agrégations $\forall$, chacune notée $A_{p\text{ME}},\,p=2$

Chaque cercle marqué $A_{p\text{ME}},\,p=2,\,\forall x_+$ (ou $\forall x_-$) représente une agrégation `pMeanError` avec $p=2$, correspondant au quantificateur `Forall` construit avec `ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")`.

Chacun de ces deux cercles prend l'ensemble des sorties individuelles obtenues à l'étape précédente pour son groupe de points, et les condense en un seul nombre. Le cercle du haut calcule ainsi la valeur de vérité de la formule close $\forall x_+\,A(x_+)$ ; celui du bas calcule celle de $\forall x_-\,\lnot A(x_-)$. En sortie de cette étape, on obtient exactement deux nombres, un par axiome de $\mathcal{K}$, plus aucun vecteur.

#### Étape 6 : l'agrégation finale, $A_{p\text{ME}},\,p=2,\,\mathbf{A}_\phi$

Ce troisième cercle reçoit en entrée les deux nombres produits à l'étape précédente, un par axiome, et les agrège à leur tour en un seul nombre final. C'est l'opérateur `SatAgg`.

La différence de notation à remarquer : les deux premiers cercles sont indicés $\forall x_+$ et $\forall x_-$, ils agrègent sur les individus d'une variable, tandis que ce dernier cercle est indicé $\mathbf{A}_\phi$, ce qui signifie qu'il agrège sur l'ensemble des formules $\phi$ de la base de connaissances $\mathcal{K}$, pas sur des individus d'un batch. C'est la même famille d'opérateur (`pMeanError`, $p=2$), mais appliquée à un niveau différent : d'abord chaque axiome est résumé individuellement sur ses variables, puis l'ensemble des axiomes est résumé entre eux via `SatAgg`.

#### Étape 7 : la sortie finale, `sat`

Le dernier rectangle, `sat`, est le nombre unique qui sort de tout le pipeline : le niveau global de satisfaction de la base de connaissances $\mathcal{K}$, à cette étape précise de l'entraînement. C'est la valeur `sat_agg` du code, utilisée ensuite pour calculer `loss = 1. - sat_agg`.

---

### Synthèse

Le schéma se lit en trois grandes phases. D'abord, chaque point de chaque groupe traverse le même réseau $A$ pour recevoir sa propre sortie individuelle, avec une négation appliquée uniquement côté $x_-$. Ensuite, ces sorties individuelles sont résumées en un seul nombre par axiome, via le quantificateur $\forall$. Enfin, ces deux nombres, un par axiome, sont eux-mêmes résumés en un seul niveau de satisfaction global via `SatAgg`, ce nombre final pilotant l'entraînement du réseau.

In [ ]:
optimizer = torch.optim.Adam(A.parameters(), lr=0.001)

# training of the predicate A using a loss containing the satisfaction level of the knowledge base
# the objective it to maximize the satisfaction level of the knowledge base
for epoch in range(1000):
    train_loss = 0.0
    for batch_idx, (data, labels) in enumerate(train_loader):
        optimizer.zero_grad()
        # we ground the variables with current batch data
        x_A = ltn.Variable("x_A", data[torch.nonzero(labels)]) # positive examples
        x_not_A = ltn.Variable("x_not_A", data[torch.nonzero(torch.logical_not(labels))]) # negative examples
        sat_agg = SatAgg(
            Forall(x_A, A(x_A)),
            Forall(x_not_A, Not(A(x_not_A)))
        )
        loss = 1. - sat_agg
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss = train_loss / len(train_loader)

    # we print metrics every 20 epochs of training
    if epoch % 20 == 0:
        print(" epoch %d | loss %.4f | Train Sat %.3f | Test Sat %.3f | Train Acc %.3f | Test Acc %.3f"
              %(epoch, train_loss, compute_sat_level(train_loader), compute_sat_level(test_loader),
                    compute_accuracy(train_loader), compute_accuracy(test_loader)))

Les variables $x_+$ et $x_-$ sont groundées batch par batch, avec de nouvelles données provenant du chargeur à chaque itération. C'est exactement ce que représente la notation $\mathcal{G}_{x\leftarrow B}(\phi(x))$, où $B$ est le mini-batch fourni par le chargeur.

`SatAgg` prend en entrée les deux axiomes et renvoie une seule valeur de vérité, interprétée comme le niveau de satisfaction de la base de connaissances, exactement le mécanisme détaillé plus haut.

Après 800 epochs, la précision sur les données de test avoisine 1, ce qui montre la capacité de LTN à apprendre cette tâche de classification binaire en n'utilisant que la satisfaction d'une base de connaissances comme objectif.

On interroge maintenant le prédicat $A$ à la fois sur les données d'entraînement et sur les données de test. Le graphique qui suit permet de visualiser l'étendue de la généralisation obtenue.

In [ ]:
nr_samples_train = 50

fig = plt.figure(figsize=(9, 11))
plt.subplots_adjust(wspace=0.2, hspace=0.3)

ax = plt.subplot2grid((3,8), (0,2), colspan=4)
ax.set_title("vérité terrain")
ax.scatter(dataset[labels_dataset][:,0], dataset[labels_dataset][:,1], label='A')
ax.scatter(dataset[torch.logical_not(labels_dataset)][:,0], dataset[torch.logical_not(labels_dataset)][:,1], label='~A')
ax.legend()

# données d'entraînement
x = ltn.Variable("x", dataset[:nr_samples_train])
fig.add_subplot(3, 2, 3)
result = A(x)
plt.title("A(x) - données d'entraînement")
plt.scatter(dataset[:nr_samples_train,0], dataset[:nr_samples_train,1], c=result.value.detach().numpy().squeeze())
plt.colorbar()

fig.add_subplot(3, 2, 4)
result = Not(A(x))
plt.title("~A(x) - données d'entraînement")
plt.scatter(dataset[:nr_samples_train,0], dataset[:nr_samples_train,1], c=result.value.detach().numpy().squeeze())
plt.colorbar()

# données de test
x = ltn.Variable("x", dataset[nr_samples_train:])
fig.add_subplot(3, 2, 5)
result = A(x)
plt.title("A(x) - données de test")
plt.scatter(dataset[nr_samples_train:,0], dataset[nr_samples_train:,1], c=result.value.detach().numpy().squeeze())
plt.colorbar()

fig.add_subplot(3, 2, 6)
result = Not(A(x))
plt.title("~A(x) - données de test")
plt.scatter(dataset[nr_samples_train:,0], dataset[nr_samples_train:,1], c=result.value.detach().numpy().squeeze())
plt.colorbar()
plt.savefig("ex_binary_testing.pdf")
plt.show()

### Discussion

Ce premier exemple, bien que simple, illustre pas à pas le processus complet d'utilisation de LTN dans un cadre élémentaire.

Dans ce cas précis, la base de connaissances n'a servi qu'à transmettre au modèle sa vérité terrain : le prédicat $A$ doit être maximisé quand l'entrée est un exemple positif, minimisé quand elle est négative. Cet objectif rappelle celui d'une entropie croisée binaire, la fonction de perte standard de tout classifieur binaire, mais seulement dans son intention : dans sa forme exacte, la loss LTN utilisée ici équivaut en réalité à une racine d'erreur quadratique moyenne (RMSE), pas à une entropie croisée logarithmique.

<details>
<summary><b>Démonstration : dérivation complète de la loss LTN et comparaison à la BCE</b></summary>

**La loss standard, l'entropie croisée binaire.** Pour un point $x$ de vraie étiquette $y\in\{0,1\}$, avec $A(x)$ la probabilité prédite :

$$\text{BCE}(x,y) = -\big[y\log A(x) + (1-y)\log(1-A(x))\big]$$

Sur l'ensemble du batch, moyennée :

$$L_{\text{BCE}} = -\frac{1}{N}\left(\sum_{x\in x_+}\log A(x) + \sum_{x\in x_-}\log(1-A(x))\right)$$

**La loss LTN, dérivée explicitement.** Repartons de `loss = 1 - SatAgg(Forall(x_+,A(x_+)), Forall(x_-,Not(A(x_-))))`, avec `pMeanError`, $p=2$, pour `Forall` et pour `SatAgg`.

Notons $s_+=\text{Forall}(x_+,A(x_+))$ et $s_-=\text{Forall}(x_-,\lnot A(x_-))$. Par définition de `pMeanError` :

$$1-s_+ = \left(\frac{1}{N_+}\sum_{x\in x_+}(1-A(x))^p\right)^{1/p} \qquad 1-s_- = \left(\frac{1}{N_-}\sum_{x\in x_-}A(x)^p\right)^{1/p}$$

`SatAgg` applique `pMeanError` une seconde fois, sur ces deux valeurs :

$$1-\text{SatAgg}(s_+,s_-) = \left(\frac{1}{2}\big[(1-s_+)^p+(1-s_-)^p\big]\right)^{1/p} = \left(\frac{1}{2}\left[\frac{1}{N_+}\sum_{x_+}(1-A(x))^p + \frac{1}{N_-}\sum_{x_-}A(x)^p\right]\right)^{1/p}$$

Avec $p=2$, la loss finale devient :

$$L_{\text{LTN}} = \sqrt{\frac{1}{2}\left[\frac{1}{N_+}\sum_{x_+}(1-A(x))^2 + \frac{1}{N_-}\sum_{x_-}A(x)^2\right]}$$

C'est une erreur quadratique moyenne (MSE), pas une entropie croisée. Le terme $(1-A(x))^2$ pénalise l'écart entre $A(x)$ et la cible $1$ de façon quadratique, exactement comme une MSE classique, alors que la BCE pénalise le même écart via un logarithme.
</details>

Au-delà de cette comparaison, l'enseignement général à retenir est le suivant : des méthodes comme LTN apportent une réelle valeur ajoutée lorsque les connaissances logiques codifient une information nouvelle par rapport à la vérité terrain, c'est-à-dire lorsque l'information provenant des données et celle provenant des connaissances sont complémentaires plutôt que redondantes. Dans cet exemple, les 100 points sont tous étiquetés, la connaissance logique ne fait donc que reformuler une information déjà entièrement contenue dans les données. C'est un choix pédagogique volontairement simple pour introduire la mécanique de LTN, mais qui ne montre pas encore le plein potentiel de l'approche, celui-ci apparaissant plutôt dans des cas comme celui de la classification par proximité, où très peu de points étaient étiquetés et où la règle logique de proximité apportait une information structurelle absente des données brutes.

Les tutoriels suivants montreront comment le langage de LTN permet de résoudre des problèmes progressivement plus complexes, en combinant apprentissage et raisonnement.